# Notebook 1: Carga, Exploración y Limpieza de Datos
## Reto 2: Seguridad y Obras Públicas — Bogotá
### Concurso de Datos por la Seguridad — CCB

---

**Objetivo:** Cargar, explorar y limpiar las fuentes de datos principales para el análisis de la relación entre obras de transporte público y percepción de seguridad.

**Fuentes principales (por prioridad para el Reto 2):**
1. 🔴 **Encuesta de Corredores Estratégicos 2024** — 2,105 establecimientos en Metro PLM y Calle 13
2. 🟠 **ECN (Clima de los Negocios) 2020–2024** — Percepción de seguridad empresarial, victimización
3. 🟡 **EMU (Movilidad y Entornos Urbanos) 2024** — Expectativas del metro, seguridad, entorno
4. 🟢 **EPV (Percepción y Victimización) 2021–2025** — Percepción ciudadana, victimización, GPS
5. 🔵 **RNMC (Registro Nacional de Medidas Correctivas)** — ~2.9M comparendos 2020–2025
6. ⚪ **Base Registro Mercantil 2024–2025** — Universo empresarial para MiPymes

In [1]:
# =============================================================================
# 0. CONFIGURACIÓN INICIAL
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

# =============================================================================
# RUTAS — Ajustar según la ubicación de la carpeta 'BD concurso'
# =============================================================================
# Opción 1: Ruta relativa (si el notebook está dentro de /codigo/)
BASE_DIR = os.path.join('..') 

# Opción 2: Ruta absoluta (descomentar y ajustar si es necesario)
# BASE_DIR = r'C:\Users\TU_USUARIO\Downloads\BD concurso'

# Verificar que la carpeta existe
assert os.path.isdir(BASE_DIR), f"No se encontró la carpeta base: {BASE_DIR}"
print(f"Carpeta base: {os.path.abspath(BASE_DIR)}")
print(f"Contenido: {os.listdir(BASE_DIR)}")

Carpeta base: c:\Users\David\Desktop\Universidad\Semestre 2026-10\DATAJAM CCB\DATAJAM-CCB
Contenido: ['.git', '.venv', 'Bases de datos', 'Código.ipynb', 'Códigos Santiago', 'requirements.txt']


---
## 1. ENCUESTA DE CORREDORES ESTRATÉGICOS 2024
**La fuente más importante para el Reto 2**

- 2,105 establecimientos comerciales en dos corredores:
  - **Calle 13** (903 establecimientos, 3 tramos)
  - **Primera Línea del Metro — PLM** (1,202 establecimientos, 16 tramos)
- Variables clave: percepción de seguridad, victimización por obras, impacto en negocios
- Variable `TRAMO`: 1 = Calle 13, 2 = Metro PLM
- Variables `PRIMERA` (16 segmentos Metro) y `CALL13` (3 segmentos Calle 13)

In [ ]:
# =============================================================================
# 1.1 Cargar datos de Corredores Estratégicos
# =============================================================================
ruta_corredores = os.path.join(
    BASE_DIR, 
    'Encuesta corredores estratégicos', 
    'Base corredores estratégicos anonimizada.xlsx'
)

# El archivo tiene encabezados multi-fila. La fila 3 (index 2) tiene los códigos cortos.
# Leer con header en fila 2 (0-indexed)
corredores_raw = pd.read_excel(ruta_corredores, sheet_name='NUM', header=2)

print(f"Dimensiones: {corredores_raw.shape}")
print(f"\nPrimeras columnas: {list(corredores_raw.columns[:20])}")
corredores_raw.head(3)

In [ ]:
# =============================================================================
# 1.2 Seleccionar variables relevantes para Reto 2
# =============================================================================

# Variables de identificación y perfil
vars_id = ['NUMERO', 'TRAMO', 'PRIMERA', 'CALL13', 'FECHA', 'F5', 'F6']

# Variables de empleo y transporte
vars_empleo = ['P1', 'P2', 'P3']  # Total empleados, mujeres, jóvenes
vars_transporte_emp = ['P4A', 'P4B', 'P4C', 'P4D', 'P4E']  # % por modo

# Variables de satisfacción con el entorno (CLAVE)
vars_satisfaccion = [
    'P33A',  # Contaminación aire
    'P33B',  # Ruido
    'P33C',  # Andenes
    'P33D',  # Estado vías
    'P33E',  # Ciclorrutas
    'P33F',  # Seguridad vial
    'P33G',  # SEGURIDAD (delincuencia)
    'P33H',  # Zonas verdes
    'P33J',  # Estacionamiento
    'P33K',  # Transporte público
    'P33L',  # Congestión
    'P34',   # Calidad espacio público
]

# Opinión sobre el proyecto de corredor
vars_opinion = ['P37', 'P40']  # Acuerdo con el plan, preocupación por duración

# Seguridad y victimización (CLAVE)
vars_seguridad = [
    'P43',      # Seguridad desde que comenzaron obras: mejoró/empeoró/igual
    'P43_1A',   # Señalización
    'P43_1B',   # Iluminación calles/andenes
    'P43_1C',   # Manejo de residuos
    'P43_1D',   # Reubicación estaciones SITP/TM
    'P43_1E',   # Espacios peatonales adecuados
    'P43_1F',   # Presencia de habitantes de calle
    'P43_1G',   # Vigilancia en zonas de obra
]

# Victimización por delitos asociados a obras
vars_victimizacion = [
    'P44_1',  # Hurto al establecimiento
    'P44_2',  # Hurto a personas
    'P44_3',  # Extorsión
    'P44_4',  # Acoso sexual callejero
    'P44_5',  # Violencia de género
    'P44_6',  # Lesiones personales
    'P44_7',  # Situaciones de convivencia
    'P44_8',  # Ninguno
]

# Soluciones de seguridad
vars_soluciones = ['P45_1', 'P45_2', 'P45_3', 'P45_4', 'P45_5', 'P45_6', 'P45_7']

# Impacto económico
vars_economico = ['P30', 'P31', 'P32']  # Uso espacio público, competencia informal

# Demografía
vars_demo = ['D1', 'D2', 'D3']

# Construir lista completa
vars_corredores = (vars_id + vars_empleo + vars_transporte_emp + vars_satisfaccion + 
                   vars_opinion + vars_seguridad + vars_victimizacion + 
                   vars_soluciones + vars_economico + vars_demo)

# Filtrar solo las que existen en el DataFrame
vars_existentes = [v for v in vars_corredores if v in corredores_raw.columns]
vars_faltantes = [v for v in vars_corredores if v not in corredores_raw.columns]
if vars_faltantes:
    print(f"⚠️ Variables no encontradas: {vars_faltantes}")

corredores = corredores_raw[vars_existentes].copy()
print(f"\n✅ Dataset de corredores: {corredores.shape}")
corredores.info()

In [ ]:
# =============================================================================
# 1.3 Exploración y limpieza inicial
# =============================================================================

# Distribución por corredor
print("=" * 60)
print("DISTRIBUCIÓN POR CORREDOR (TRAMO)")
print("=" * 60)
tramo_labels = {1: 'Calle 13', 2: 'Primera Línea Metro (PLM)'}
print(corredores['TRAMO'].map(tramo_labels).value_counts())

# Distribución por segmento del metro
print("\n" + "=" * 60)
print("SEGMENTOS DEL METRO (PRIMERA)")
print("=" * 60)
print(corredores.loc[corredores['TRAMO'] == 2, 'PRIMERA'].value_counts().sort_index())

# Distribución por segmento Calle 13
print("\n" + "=" * 60)
print("SEGMENTOS CALLE 13 (CALL13)")
print("=" * 60)
print(corredores.loc[corredores['TRAMO'] == 1, 'CALL13'].value_counts().sort_index())

In [ ]:
# =============================================================================
# 1.4 Análisis de valores faltantes
# =============================================================================

# Porcentaje de valores faltantes por variable
missing = corredores.isnull().mean().sort_values(ascending=False) * 100
missing_significativo = missing[missing > 0]

if len(missing_significativo) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing_significativo) * 0.3)))
    missing_significativo.plot(kind='barh', ax=ax, color='coral')
    ax.set_xlabel('% Valores Faltantes')
    ax.set_title('Valores Faltantes — Corredores Estratégicos')
    plt.tight_layout()
    plt.show()
else:
    print("✅ No hay valores faltantes significativos")

print(f"\nResumen de nulos:\n{corredores.isnull().sum()[corredores.isnull().sum() > 0]}")

In [ ]:
# =============================================================================
# 1.5 Crear variables derivadas
# =============================================================================

# Etiquetas legibles para corredores
corredores['corredor_nombre'] = corredores['TRAMO'].map({
    1: 'Calle 13',
    2: 'Primera Línea Metro'
})

# Tamaño de empresa (MiPyme)
corredores['tamano_empresa'] = corredores['F6'].map({
    1: 'Micro',
    2: 'Pequeña',
    3: 'Mediana',
    4: 'Grande'
})

# Es MiPyme? (Micro, Pequeña o Mediana)
corredores['es_mipyme'] = corredores['F6'].isin([1, 2, 3]).astype(int)

# Seguridad empeoró (dummy)
# P43: 1=Mejoró, 2=Empeoró, 3=Igual
corredores['seguridad_empeoro'] = (corredores['P43'] == 2).astype(int)
corredores['seguridad_mejoro'] = (corredores['P43'] == 1).astype(int)

# Fue víctima de algún delito (cualquiera de P44_1 a P44_7 == 1)
cols_victima = [c for c in vars_victimizacion if c in corredores.columns and c != 'P44_8']
corredores['fue_victima'] = corredores[cols_victima].apply(
    lambda row: 1 if row.sum() > 0 else 0, axis=1
)

# Índice de satisfacción con el entorno (promedio de P33A–P33L)
cols_satisf = [c for c in vars_satisfaccion if c in corredores.columns]
corredores['indice_satisfaccion'] = corredores[cols_satisf].mean(axis=1)

print("✅ Variables derivadas creadas")
print(f"\nMiPymes: {corredores['es_mipyme'].mean()*100:.1f}%")
print(f"Seguridad empeoró: {corredores['seguridad_empeoro'].mean()*100:.1f}%")
print(f"Fue víctima: {corredores['fue_victima'].mean()*100:.1f}%")

In [ ]:
# =============================================================================
# 1.6 Visualización exploratoria rápida
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Percepción de seguridad por corredor
ax1 = axes[0, 0]
seg_corredor = corredores.groupby('corredor_nombre')['P43'].value_counts(normalize=True).unstack()
seg_labels = {1: 'Mejoró', 2: 'Empeoró', 3: 'Igual'}
seg_corredor = seg_corredor.rename(columns=seg_labels)
seg_corredor.plot(kind='bar', ax=ax1, color=['green', 'red', 'gray'])
ax1.set_title('Percepción de seguridad desde inicio de obras')
ax1.set_ylabel('Proporción')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0)
ax1.legend(title='Seguridad')

# 2. Victimización por corredor
ax2 = axes[0, 1]
vic_corredor = corredores.groupby('corredor_nombre')['fue_victima'].mean() * 100
vic_corredor.plot(kind='bar', ax=ax2, color=['steelblue', 'coral'])
ax2.set_title('% Establecimientos víctimas de delito')
ax2.set_ylabel('% Víctimas')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)

# 3. Satisfacción con seguridad (P33G) por corredor
ax3 = axes[1, 0]
if 'P33G' in corredores.columns:
    corredores.boxplot(column='P33G', by='corredor_nombre', ax=ax3)
    ax3.set_title('Satisfacción con seguridad (P33G)')
    ax3.set_ylabel('Escala 1-5')
    ax3.set_xlabel('')
    plt.suptitle('')  # Quitar título automático de boxplot

# 4. Distribución por tamaño de empresa
ax4 = axes[1, 1]
corredores['tamano_empresa'].value_counts().plot(kind='pie', ax=ax4, autopct='%1.1f%%')
ax4.set_title('Distribución por tamaño de empresa')
ax4.set_ylabel('')

plt.tight_layout()
plt.savefig('fig_01_exploracion_corredores.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Figura guardada: fig_01_exploracion_corredores.png")

---
## 2. ENCUESTA DE CLIMA DE LOS NEGOCIOS (ECN) 2024
**Percepción empresarial de seguridad y victimización**

- 2,549 empresas en Bogotá y Cundinamarca
- Variables clave: P56 (percepción seguridad), P57 (tendencia), P58 (victimización)
- Variable de tamaño de empresa: F5 (Micro/Pequeña/Mediana/Grande)
- Sección P33: ambiente de negocios (25 dimensiones)

In [ ]:
# =============================================================================
# 2.1 Cargar ECN 2024
# =============================================================================
ruta_ecn_2024 = os.path.join(
    BASE_DIR,
    'ECN_Encuesta Clima de los Negocios', '2024',
    'Encuesta Clima de negocios 2024 final anonimizada.xlsx'
)

ecn_2024 = pd.read_excel(ruta_ecn_2024, sheet_name='Cod')
print(f"ECN 2024 — Dimensiones: {ecn_2024.shape}")

# Variables más importantes para Reto 2
vars_ecn = [
    'NUMERO', 'REGION', 'F4', 'F5',  # ID, Región, Sector, Tamaño
    'P32',                             # Clima de inversión (1-5)
    'P33_C',                           # Infraestructura urbana (1-5)
    'P33_H',                           # Condiciones de seguridad (1-5)
    'P33_S',                           # Transporte público/privado (1-5)
    'P33_W',                           # Tiempos de desplazamiento (1-5)
    'P56',                             # Percepción: Bogotá segura/insegura
    'P56_1',                           # Escala seguridad 1-5
    'P57',                             # Tendencia inseguridad vs año anterior
    'P58',                             # Fue víctima de delito
    'P60',                             # Denunció
    'FACTOR_TOTAL',                    # Factor de expansión
]

# Variables de victimización por tipo
vars_ecn_delitos = [c for c in ecn_2024.columns if c.startswith('P59_')]
# Variables de costos de inseguridad
vars_ecn_costos = [c for c in ecn_2024.columns if c.startswith('P61B_')]
# Variables de prevención
vars_ecn_prevencion = [c for c in ecn_2024.columns if c.startswith('P61A_')]

vars_ecn_total = vars_ecn + vars_ecn_delitos + vars_ecn_costos + vars_ecn_prevencion
vars_ecn_existentes = [v for v in vars_ecn_total if v in ecn_2024.columns]

ecn = ecn_2024[vars_ecn_existentes].copy()
ecn['anio'] = 2024

print(f"\n✅ ECN 2024 filtrado: {ecn.shape}")
ecn.head()

In [ ]:
# =============================================================================
# 2.2 Cargar ECN años anteriores para análisis temporal
# =============================================================================
ecn_historico = []

for anio in [2020, 2021, 2022]:
    ruta = os.path.join(
        BASE_DIR,
        'ECN_Encuesta Clima de los Negocios', str(anio),
        f'Base_{anio}.xlsx'
    )
    if os.path.exists(ruta):
        try:
            df = pd.read_excel(ruta)
            df['anio'] = anio
            ecn_historico.append(df)
            print(f"  ✅ ECN {anio}: {df.shape}")
        except Exception as e:
            print(f"  ⚠️ ECN {anio}: Error al cargar — {e}")

# ECN 2023 tiene diferente nombre de archivo
ruta_2023 = os.path.join(
    BASE_DIR,
    'ECN_Encuesta Clima de los Negocios', '2023',
    'Base CLIMA DE NEGOCIOS 2023 anonimizada.xlsx'
)
if os.path.exists(ruta_2023):
    try:
        df_2023 = pd.read_excel(ruta_2023, sheet_name=0)  # Primer hoja
        df_2023['anio'] = 2023
        ecn_historico.append(df_2023)
        print(f"  ✅ ECN 2023: {df_2023.shape}")
    except Exception as e:
        print(f"  ⚠️ ECN 2023: Error al cargar — {e}")

print(f"\n📦 Total de años cargados: {len(ecn_historico) + 1} (incluyendo 2024)")

In [ ]:
# =============================================================================
# 2.3 Limpieza y variables derivadas ECN 2024
# =============================================================================

# Tamaño de empresa
ecn['tamano_empresa'] = ecn['F5'].map({
    1: 'Micro', 2: 'Pequeña', 3: 'Mediana', 4: 'Grande'
})
ecn['es_mipyme'] = ecn['F5'].isin([1, 2, 3]).astype(int)

# Sector económico
ecn['sector'] = ecn['F4'].map({
    1: 'Industria', 2: 'Comercio', 3: 'Servicios'
})

# Solo Bogotá (REGION == 1)
ecn_bogota = ecn[ecn['REGION'] == 1].copy()
print(f"ECN 2024 solo Bogotá: {ecn_bogota.shape[0]} empresas")

# Estadísticas clave
print(f"\n--- PERCEPCIÓN DE SEGURIDAD (P56) ---")
if 'P56' in ecn_bogota.columns:
    print(ecn_bogota['P56'].value_counts(normalize=True).round(3) * 100)

print(f"\n--- FUE VÍCTIMA (P58) ---")
if 'P58' in ecn_bogota.columns:
    print(ecn_bogota['P58'].value_counts(normalize=True).round(3) * 100)

print(f"\n--- SEGURIDAD POR TAMAÑO DE EMPRESA ---")
if 'P56_1' in ecn_bogota.columns:
    print(ecn_bogota.groupby('tamano_empresa')['P56_1'].mean().round(2))

---
## 3. ENCUESTA DE MOVILIDAD Y ENTORNOS URBANOS (EMU) 2024
**Percepción ciudadana sobre movilidad, seguridad y expectativas del Metro**

- 18,865 encuestados en las 19 localidades de Bogotá
- 843 variables
- Variables clave: P131 (seguridad en entorno), P133 (expectativas Metro), P134 (medidas de seguridad)

In [ ]:
# =============================================================================
# 3.1 Cargar EMU 2024
# =============================================================================
ruta_emu = os.path.join(
    BASE_DIR,
    'Encuesta de movilidad y entornos urbanos',
    'Resultados Encuesta de movilidad y entornos urbanos 2024 anonimizada.xlsx'
)

emu_raw = pd.read_excel(ruta_emu)
print(f"EMU 2024 — Dimensiones: {emu_raw.shape}")

# Variables de interés para Reto 2
vars_emu_id = ['ID', 'Localidad', 'UPL']

# Seguridad en el entorno urbano (escala 1-5, 22 elementos)
vars_emu_seguridad = [f'P131_{i}' for i in range(1, 23)]

# Expectativas sobre el Metro (13 ítems Si/No)
vars_emu_metro = [f'P133_{i}' for i in range(1, 14)]

# Medidas más importantes para mejorar seguridad en movilidad
vars_emu_medidas = [f'P134_{i}' for i in range(1, 13)]

# Satisfacción con elementos urbanos (27 ítems)
vars_emu_satisf = [f'P130_{i}' for i in range(1, 28)]

# Percepción de seguridad en transporte público
vars_emu_seg_tp = ['P8_1', 'P8_2', 'P8_3']  # Dentro del bus, estaciones, alrededores

# Perspectivas de movilidad
vars_emu_perspectiva = ['P152']  # Movilidad mejoró o empeoró

# Demografía
vars_emu_demo = ['P800', 'P801', 'REDAD', 'P802', 'FACTOR_1', 'FACTOR_2']

# General satisfaction
vars_emu_gral = ['P129']  # Satisfacción general con Bogotá

vars_emu_total = (vars_emu_id + vars_emu_seguridad + vars_emu_metro + vars_emu_medidas + 
                  vars_emu_satisf + vars_emu_seg_tp + vars_emu_perspectiva + 
                  vars_emu_demo + vars_emu_gral)

vars_emu_existentes = [v for v in vars_emu_total if v in emu_raw.columns]
vars_emu_faltantes = [v for v in vars_emu_total if v not in emu_raw.columns]
if vars_emu_faltantes:
    print(f"⚠️ Variables no encontradas: {vars_emu_faltantes[:10]}...")

emu = emu_raw[vars_emu_existentes].copy()
print(f"\n✅ EMU 2024 filtrado: {emu.shape}")

In [ ]:
# =============================================================================
# 3.2 Exploración de expectativas del Metro
# =============================================================================
metro_labels = {
    'P133_1': 'Reducirá trancones',
    'P133_2': 'Viajes más rápidos',
    'P133_3': 'Costos transporte bajarán',
    'P133_4': 'Menos transbordos',
    'P133_5': 'Movilidad más segura',
    'P133_6': 'Menor contaminación',
    'P133_7': 'Nuevos negocios',
    'P133_8': 'Generación de empleo',
    'P133_9': 'Aumento valor inmuebles',
    'P133_10': 'Pérdida valor por ruido',
    'P133_11': 'Muchos transbordos',
    'P133_12': 'Empezaré a usar TP',
    'P133_13': 'Otro',
}

# Calcular % de respuestas "Sí" para cada expectativa
vars_metro_exist = [v for v in vars_emu_metro if v in emu.columns]
expectativas = emu[vars_metro_exist].apply(lambda x: (x == 1).mean() * 100)
expectativas.index = [metro_labels.get(v, v) for v in expectativas.index]

fig, ax = plt.subplots(figsize=(10, 6))
expectativas.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('% que responde Sí')
ax.set_title('Expectativas sobre la Primera Línea del Metro — EMU 2024')
plt.tight_layout()
plt.savefig('fig_02_expectativas_metro.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Figura guardada: fig_02_expectativas_metro.png")

---
## 4. EPV — ENCUESTA DE PERCEPCIÓN Y VICTIMIZACIÓN 2025
**Percepción ciudadana y victimización con coordenadas GPS**

- 20,038 encuestados
- 503 variables
- Incluye GPS y localización precisa
- Tiene datos de 2021 a 2025 (panel potencial por localidad)

In [ ]:
# =============================================================================
# 4.1 Cargar EPV 2025
# =============================================================================
ruta_epv = os.path.join(
    BASE_DIR,
    'EPV_Encuesta de Percepción y Victimización', '2025',
    'Base Encuesta de Percepción Victimización 2025 anonimizada.xlsx'
)

epv_raw = pd.read_excel(ruta_epv)
print(f"EPV 2025 — Dimensiones: {epv_raw.shape}")

# Variables de interés
vars_epv_id = ['Localidad', 'UPL']
vars_epv_demo = ['A1', 'A2', 'A3', 'A4']  # Género, edad, educación, estrato
vars_epv_percepcion = [
    'P102',    # Barrio seguro/inseguro
    'P102_1',  # Escala seguridad barrio 1-5
    'P103_1',  # Escala seguridad Bogotá 1-5
    'P106',    # Tendencia inseguridad
    'P111',    # Seguridad TransMilenio 1-5
]
vars_epv_victima = ['P203']  # ¿Fue víctima?
vars_epv_gps = ['_Capture_gps_latitude', '_Capture_gps_longitude']
vars_epv_peso = ['FEX']  # Factor de expansión

vars_epv_total = vars_epv_id + vars_epv_demo + vars_epv_percepcion + vars_epv_victima + vars_epv_gps + vars_epv_peso
vars_epv_existentes = [v for v in vars_epv_total if v in epv_raw.columns]

epv = epv_raw[vars_epv_existentes].copy()
epv['anio'] = 2025
print(f"\n✅ EPV 2025 filtrado: {epv.shape}")
print(f"Registros con GPS: {epv['_Capture_gps_latitude'].notna().sum() if '_Capture_gps_latitude' in epv.columns else 'N/A'}")

In [ ]:
# =============================================================================
# 4.2 Cargar EPV años anteriores (para análisis temporal)
# =============================================================================
epv_panel = []

for anio in [2021, 2022, 2023, 2024]:
    carpeta = os.path.join(BASE_DIR, 'EPV_Encuesta de Percepción y Victimización', str(anio))
    if os.path.isdir(carpeta):
        archivos_xlsx = [f for f in os.listdir(carpeta) if f.endswith('.xlsx') and 'Base' in f]
        if archivos_xlsx:
            ruta = os.path.join(carpeta, archivos_xlsx[0])
            try:
                df = pd.read_excel(ruta)
                df['anio'] = anio
                epv_panel.append(df)
                print(f"  ✅ EPV {anio}: {df.shape}")
            except Exception as e:
                print(f"  ⚠️ EPV {anio}: Error — {e}")

print(f"\n📦 Años EPV cargados: {[df['anio'].iloc[0] for df in epv_panel]}")

---
## 5. RNMC — REGISTRO NACIONAL DE MEDIDAS CORRECTIVAS
**~2.9M comparendos policiales 2020–2025 con localización**

- Datos administrativos de infracciones al Código de Convivencia
- 16 variables: año, mes, día, franja horaria, localidad, UPZ, tipo de infracción
- Permite construir indicadores objetivos de actividad policial/infraccional por zona

In [ ]:
# =============================================================================
# 5.1 Cargar RNMC
# =============================================================================
ruta_rnmc = os.path.join(BASE_DIR, 'RNMC', 'RNMC_anon_V2.csv')

# Archivo grande (~2.9M filas). Cargar con tipos optimizados.
dtypes_rnmc = {
    'ANIO': 'int16',
    'MES': 'int8',
    'DIA_SEMANA': 'category',
    'FRANJA_HORARIA': 'category',
    'COD_LOCALIDAD': 'int16',
    'NOMBRE_LOCALIDAD': 'category',
    'COD_UPZ': 'str',
    'NOMBRE_UPZ': 'category',
    'TITULO': 'category',
    'DESCRIPCION_TITULO': 'category',
}

rnmc = pd.read_csv(ruta_rnmc, dtype=dtypes_rnmc, low_memory=False)
print(f"RNMC — Dimensiones: {rnmc.shape}")
print(f"Memoria: {rnmc.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nRegistros por año:")
print(rnmc['ANIO'].value_counts().sort_index())

In [ ]:
# =============================================================================
# 5.2 Filtrar infracciones relevantes para seguridad
# =============================================================================

# Ver categorías de título (tipo de infracción)
print("TIPOS DE INFRACCIÓN (TITULO):")
print(rnmc['DESCRIPCION_TITULO'].value_counts())

# Infracciones de seguridad
rnmc_seguridad = rnmc[
    rnmc['DESCRIPCION_TITULO'].str.contains('SEGURIDAD', case=False, na=False)
].copy()
print(f"\n✅ Infracciones de seguridad: {len(rnmc_seguridad):,}")

# Infracciones de movilidad
rnmc_movilidad = rnmc[
    rnmc['DESCRIPCION_TITULO'].str.contains('MOVILIDAD', case=False, na=False)
].copy()
print(f"✅ Infracciones de movilidad: {len(rnmc_movilidad):,}")

In [ ]:
# =============================================================================
# 5.3 Agregar indicadores por localidad y año
# =============================================================================

# Total comparendos por localidad y año
rnmc_localidad = rnmc.groupby(['ANIO', 'NOMBRE_LOCALIDAD']).size().reset_index(name='total_comparendos')

# Comparendos de seguridad por localidad y año
rnmc_seg_loc = rnmc_seguridad.groupby(['ANIO', 'NOMBRE_LOCALIDAD']).size().reset_index(name='comparendos_seguridad')

# Comparendos de movilidad por localidad y año
rnmc_mov_loc = rnmc_movilidad.groupby(['ANIO', 'NOMBRE_LOCALIDAD']).size().reset_index(name='comparendos_movilidad')

# Merge
rnmc_indicadores = rnmc_localidad.merge(rnmc_seg_loc, on=['ANIO', 'NOMBRE_LOCALIDAD'], how='left')
rnmc_indicadores = rnmc_indicadores.merge(rnmc_mov_loc, on=['ANIO', 'NOMBRE_LOCALIDAD'], how='left')
rnmc_indicadores = rnmc_indicadores.fillna(0)

print(f"✅ Indicadores RNMC por localidad-año: {rnmc_indicadores.shape}")
rnmc_indicadores.head(10)

In [ ]:
# =============================================================================
# 5.4 Indicadores por UPZ (más granular)
# =============================================================================

# Total y seguridad por UPZ y año
rnmc_upz = rnmc.groupby(['ANIO', 'NOMBRE_UPZ', 'NOMBRE_LOCALIDAD']).size().reset_index(name='total_comparendos')
rnmc_seg_upz = rnmc_seguridad.groupby(['ANIO', 'NOMBRE_UPZ']).size().reset_index(name='comparendos_seguridad')

rnmc_upz = rnmc_upz.merge(rnmc_seg_upz, on=['ANIO', 'NOMBRE_UPZ'], how='left').fillna(0)
rnmc_upz['tasa_seguridad'] = rnmc_upz['comparendos_seguridad'] / rnmc_upz['total_comparendos']

print(f"✅ Indicadores RNMC por UPZ-año: {rnmc_upz.shape}")
print(f"\nUPZs únicas: {rnmc_upz['NOMBRE_UPZ'].nunique()}")

---
## 6. GUARDAR DATOS LIMPIOS

Guardamos los datos procesados en formato `.parquet` o `.csv` para usarlos en el siguiente notebook.

In [ ]:
# =============================================================================
# 6.1 Guardar todos los datasets limpios
# =============================================================================
output_dir = os.path.join('.', 'datos_limpios')
os.makedirs(output_dir, exist_ok=True)

# Corredores
corredores.to_csv(os.path.join(output_dir, 'corredores_limpio.csv'), index=False)
print(f"✅ corredores_limpio.csv — {corredores.shape}")

# ECN 2024
ecn_bogota.to_csv(os.path.join(output_dir, 'ecn_2024_bogota.csv'), index=False)
print(f"✅ ecn_2024_bogota.csv — {ecn_bogota.shape}")

# EMU 2024
emu.to_csv(os.path.join(output_dir, 'emu_2024_filtrado.csv'), index=False)
print(f"✅ emu_2024_filtrado.csv — {emu.shape}")

# EPV 2025
epv.to_csv(os.path.join(output_dir, 'epv_2025_filtrado.csv'), index=False)
print(f"✅ epv_2025_filtrado.csv — {epv.shape}")

# RNMC indicadores
rnmc_indicadores.to_csv(os.path.join(output_dir, 'rnmc_indicadores_localidad.csv'), index=False)
rnmc_upz.to_csv(os.path.join(output_dir, 'rnmc_indicadores_upz.csv'), index=False)
print(f"✅ rnmc_indicadores_localidad.csv — {rnmc_indicadores.shape}")
print(f"✅ rnmc_indicadores_upz.csv — {rnmc_upz.shape}")

print(f"\n🎉 Todos los datos guardados en: {os.path.abspath(output_dir)}")

---
## Resumen del Notebook 1

| Dataset | Registros | Variables seleccionadas | Archivo de salida |
|---------|-----------|----------------------|------------------|
| Corredores Estratégicos 2024 | 2,105 | ~50+ | `corredores_limpio.csv` |
| ECN 2024 (Bogotá) | ~2,000+ | ~30+ | `ecn_2024_bogota.csv` |
| EMU 2024 | 18,865 | ~80+ | `emu_2024_filtrado.csv` |
| EPV 2025 | 20,038 | ~15 | `epv_2025_filtrado.csv` |
| RNMC (indicadores) | ~400+ | 5 | `rnmc_indicadores_*.csv` |

### Próximo paso → Notebook 02: Consolidación, Modelos Econométricos y Clustering